# Notebooks

Every section of this book is an executable notebook: prose, mathematics,
code, and the code's output live in a single document. That is a deliberate
choice, not a formatting convenience. Deep learning is an empirical subject.
The fastest way to understand a model is to train it, change something — the
learning rate, the width of a layer, the amount of data — and watch what
happens. A notebook makes that loop immediate: the explanation sits next to
the experiment, and the experiment is one keystroke from running. This is
also how most practitioners work day to day, from quick data inspection to
research prototypes, which is why the Jupyter notebook has become the shared
medium of machine learning.

Reading alone will not build the same understanding. We wrote each notebook
expecting you to *edit* it: break the code, fix it, try a variant the text
only mentions. This section explains how to run the book on your own machine
and how to work in notebooks effectively — including the one habit, *restart
and run all*, that separates reproducible notebooks from lucky ones. If you
would rather not install anything, that section shows how
to run the same notebooks free of charge on Colab or Kaggle; if you need more
hardware than your laptop offers, see that section.

## Why Notebooks?

A plain Python script re-runs from the top every time, so a slow step — say,
loading and preprocessing a dataset — repeats on every experiment. A notebook
keeps a live Python process (the *kernel*) between executions: load the data
once, then iterate on the model as often as you like while the data stays in
memory. Combined with inline plots and the ability to interleave explanation
with computation, this is what makes notebooks such an effective medium for
teaching and for research alike.

The price of that convenience is *state*. Understanding where state lives is
the key to using notebooks well.

### The Document and the Kernel

A notebook is really two things. The *document* stores cells and saved
outputs — it is what you read, share, and commit. The *kernel* holds the live
state: imported modules, variables, random-number generators, open files,
compiled programs, and accelerator memory. The document orders cells top to
bottom; the kernel only knows the order in which you *executed* them.
Confusing these two orders is the root cause of most notebook bugs.

![The document orders cells top to bottom; the kernel accumulates state in execution order. Restart and run all replays document order — the only order a reader can reproduce — and exposes the mismatch.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/tools-kernel-state.svg)

the figure shows the classic failure. During an
editing session you executed a data-loading cell first, then wrote a
model-definition cell *above* it, and everything worked because the kernel
already held `data`. The document now tells a story that never happened. A
reader who runs the cells in the order shown — or you, tomorrow, after a
restart — hits a `NameError`.

The execution counters (`In [3]`, `In [1]`, …) are historical clues, not a
dependency graph. The following pair of cells demonstrates benign,
document-ordered state: the second cell works only because the first ran.

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
samples = rng.normal(size=5)

In [ ]:
float(samples.mean())

That dependency is fine, because it follows the reading order. A cell that
depends on an assignment *below* it, or on a cell you have since deleted, is
not. Keep cells small, keep function definitions separate from experiments,
and pass data explicitly rather than mutating globals across many cells.

### Restart and Run All

**Restart kernel and run all cells** discards the kernel's accumulated state
and replays the document from a blank slate, in reading order. It is the
single most important reproducibility check a notebook offers: it catches
hidden state, missing setup, and order dependencies before another reader
does. Make it a reflex before saving, sharing, or committing a notebook.
Every notebook in this book is built and tested exactly this way — the
outputs you see on the website are produced by a clean top-to-bottom run.

## Running the Book Locally

### Setting Up

Download the notebook archive described in that section, or
clone the current sources from [GitHub](https://github.com/smolix/d2l-neu).
The archive includes CPU and GPU `uv` environment files. From the extracted
directory, create the environment once and launch JupyterLab through it:

```bash
uv sync --locked
uv run jupyter lab
```

Use the GPU lock file on a compatible NVIDIA system when the examples require
it. One subtlety matters more than any other: the *kernel* selected inside a
notebook must belong to the environment where the packages were installed. A
Jupyter server can see many kernels; its own Python process does not
determine which one executes your notebook.

### A Quick Sanity Check

Before changing code, record a compact identity check:

In [ ]:
import os
import platform
import sys

{
    "python": sys.executable,
    "version": platform.python_version(),
    "working_directory": os.getcwd(),
}

This catches the two most common setup mistakes in one cell: a kernel from
the wrong environment (`sys.executable` points somewhere unexpected) and a
notebook opened from a directory where relative data paths no longer resolve.

## Working in an Editor

### JupyterLab

[JupyterLab](https://jupyterlab.readthedocs.io/) combines a file browser,
notebook editor, terminals, a text editor, debugger support, and a view of
running kernels. The essentials are stable even as the interface evolves:

* Run a cell with `Shift+Enter`; use a terminal tab for `uv`, Git, and
  inspecting files.
* Select the kernel by environment name, then verify `sys.executable`.
* Interrupt a long computation before restarting. Restarting releases Python
  state and, normally, the accelerator memory owned by that process.
* Use **Restart Kernel and Run All Cells** before saving a result for others.
* Inspect the **Running** panel and stop kernels you no longer use — closing
  a browser tab does not necessarily stop its kernel, and an orphaned kernel
  can hold gigabytes of GPU memory.

### VS Code

[Visual Studio Code](https://code.visualstudio.com/docs/datascience/jupyter-notebooks)
edits and runs `.ipynb` files with kernel selection, a variable inspector,
cell-level debugging, and notebook-aware diffs. Open the repository as a
folder, choose the interpreter created by `uv`, and select that interpreter
as the notebook kernel.

VS Code shines when a notebook grows into a project: the same editor
navigates definitions, runs tests, formats code, and reviews Git diffs. A
good division of labor is to keep logic that deserves tests in `.py` modules
and use the notebook as the explanatory path and experiment record. This
repository also ships a VS Code extension that understands the book's
authoring workflow — switching framework views, previewing slides, syncing
edits back to the source — described in that section.

### Debugging and Timing

Use the notebook debugger when the kernel supports it, or move a small
failing call into a module and debug it in a terminal. Resist broad
`try/except` blocks that print "failed": they discard the exception type and
location that diagnosis needs. For quick measurements, IPython's magics are
built in:

In [ ]:
values = np.arange(100_000, dtype=np.float64)
%timeit values @ values

One caveat when timing accelerators: GPU operations are asynchronous, so a
host-side timer may measure only the dispatch of work, not its completion.
Synchronize explicitly (or use the framework's benchmarking utilities), warm
up compiled kernels first, and record the shapes, precision, device, and
library versions alongside the number.

## Remote Machines

Your editor, the notebook server, the kernel, and the accelerator need not
be on the same machine. A common setup runs JupyterLab next to the GPU while
you edit from a laptop.

![A local editor reaches a remote Jupyter server and kernel through an SSH tunnel; the accelerator stays attached to the remote kernel.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/tools-remote-layers.svg)

Bind the remote server to loopback and forward it over SSH:

```bash
# Remote machine
uv run jupyter lab --no-browser --ip 127.0.0.1 --port 8888
```


```bash
# Local machine
ssh -N -L 8888:127.0.0.1:8888 myserver
```

Then open the tokenized `http://127.0.0.1:8888/...` URL locally. Never bind
an unauthenticated Jupyter server to all network interfaces: anyone who can
reach the port can execute code as you. The SSH tunnel provides encryption
and access control; Jupyter's token remains a useful second boundary.

VS Code's Remote SSH goes one step further: it installs a small server-side
component so the editor UI runs locally while terminals, extensions, files,
and kernels all live on the remote machine. Confirm the status bar shows
**SSH** before selecting the Python environment — a local kernel cannot use
the remote GPU. Either way, treat remote compute as disposable and its
storage as precious: commit code and copy checkpoints off the machine before
it is rebooted, preempted, or deleted (that section).

Before sharing or committing any notebook, a final hygiene pass pays off:
restart and run all; check that setup does not rely on personal paths or
unrecorded downloads; remove secrets and private data from outputs; keep the
outputs that teach or verify something and delete noisy progress logs; and
note the environment and hardware needed to interpret any measurement.

## Summary

* Notebooks interleave prose, code, and results, and keep expensive state
  alive between experiments — that is why this book is written as notebooks,
  and why you should run and edit them rather than only read them.
* The document and the kernel hold different state in different orders;
  restart and run all is the test that they agree.
* Launch JupyterLab from the book's `uv` environment, and verify the kernel
  with `sys.executable` before trusting any result.
* JupyterLab integrates notebooks, terminals, and kernels; VS Code adds
  modules, tests, diffs, and first-class remote development.
* Use SSH tunneling or VS Code Remote SSH instead of exposing a notebook
  server to the network.

## Exercises

1. Create an intentional out-of-order dependency like the one in
   the figure, confirm that it works interactively,
   and then detect it with restart and run all.
1. Compare `sys.executable` in a terminal, a JupyterLab kernel, and a VS
   Code kernel on your machine. Explain any difference.
1. Time a matrix product with `%timeit` on CPU and, if available, on a GPU
   with and without synchronization. Explain the discrepancy.
1. Connect to a remote machine through an SSH tunnel and identify where the
   editor, server, kernel, and file system each run.